# P04A — Generator data sintesis algoritma

Menyiapkan empat data latihan ber-seed tetap untuk contoh regresi, klasifikasi, clustering dan anomali. Seluruh nilai/label berasal dari aturan simulasi, bukan pengukuran lapangan. Jalankan notebook ini lebih dahulu.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs

SEED = 41804
rng = np.random.default_rng(SEED)
# Dari akar repositori: tulis ke praktikum/bab04/data_bab4; di Colab atau folder lain: data_bab4 di direktori kerja.
out_dir = Path("praktikum/bab04/data_bab4") if Path("praktikum/bab04").is_dir() else Path("data_bab4")
out_dir.mkdir(parents=True, exist_ok=True)

## 1. Regresi sintesis kuat tekan beton
Rumus memperlihatkan pengaruh campuran dan umur dengan noise. Koefisien bukan kalibrasi material dan tidak dapat dipakai untuk desain campuran.

In [2]:
n = 1200
semen = rng.uniform(220, 480, n)
air = rng.uniform(135, 245, n)
fly_ash = rng.uniform(0, 120, n)
superplasticizer = rng.uniform(0, 18, n)
agregat_kasar = rng.uniform(850, 1150, n)
agregat_halus = rng.uniform(600, 900, n)
umur = rng.choice([3, 7, 14, 28, 56, 90], n)
rasio = air / semen
kuat_tekan = (8 + 0.055*semen - 0.085*air + 0.025*fly_ash
              + 0.55*superplasticizer + 5.2*np.log(umur)
              + rng.normal(0, 3.0, n))
beton = pd.DataFrame({
    "id_spesimen": [f"B{i:04d}" for i in range(1, n+1)],
    "semen_kg_m3": semen.round(1), "air_kg_m3": air.round(1),
    "rasio_air_semen": rasio.round(3), "fly_ash_kg_m3": fly_ash.round(1),
    "superplasticizer_kg_m3": superplasticizer.round(2),
    "agregat_kasar_kg_m3": agregat_kasar.round(1),
    "agregat_halus_kg_m3": agregat_halus.round(1), "umur_hari": umur,
    "kuat_tekan_MPa": kuat_tekan.round(2)})
beton.to_csv(out_dir / "data_sintesis_kuat_tekan.csv", index=False)
beton.head()

,id_spesimen,semen_kg_m3,air_kg_m3,rasio_air_semen,fly_ash_kg_m3,superplasticizer_kg_m3,agregat_kasar_kg_m3,agregat_halus_kg_m3,umur_hari,kuat_tekan_MPa
0,B0001,371.2,199.2,0.536,29.8,13.22,880.1,651.5,14,33.29
1,B0002,462.2,152.5,0.330,109.7,11.69,960.2,711.9,56,45.04
2,B0003,347.3,181.6,0.523,3.4,10.98,1085.5,654.3,3,23.70
3,B0004,440.2,215.4,0.489,24.2,16.87,1003.4,645.1,56,46.07
4,B0005,339.3,170.7,0.503,108.9,6.97,864.7,817.0,14,36.37


## 2. Klasifikasi sintesis prioritas inspeksi jembatan
Label simulasi merupakan aturan latihan, bukan rekomendasi manajemen aset.

In [3]:
m = 600
umur_jembatan = rng.integers(1, 101, m)
lalin_berat = rng.integers(100, 12001, m)
risiko_gerusan = rng.uniform(0, 1, m)
indeks_deteriorasi = rng.uniform(0, 1, m)
paparan_banjir = rng.uniform(0, 1, m)
skor = (.004*umur_jembatan + .0001*lalin_berat
        + .24*risiko_gerusan + .21*indeks_deteriorasi
        + .16*paparan_banjir + rng.normal(0, .10, m))
prioritas = skor > 1.10
jembatan = pd.DataFrame({
    "id_aset": [f"J{i:03d}" for i in range(m)],
    "umur_tahun": umur_jembatan, "lalin_berat_harian": lalin_berat,
    "risiko_gerusan": risiko_gerusan.round(3),
    "indeks_deteriorasi": indeks_deteriorasi.round(3),
    "paparan_banjir": paparan_banjir.round(3),
    "prioritas_sintesis": np.where(prioritas, "prioritas", "rutin")})
jembatan.to_csv(out_dir / "data_sintesis_prioritas_jembatan.csv", index=False)
jembatan["prioritas_sintesis"].value_counts(normalize=True)

prioritas_sintesis
prioritas    0.513333
rutin        0.486667
Name: proportion, dtype: float64

## 3. Clustering sintesis segmen perkerasan
Kolom `label_pembangkit` hanya untuk diskusi pasca-analisis. Keluarkan kolom itu dan ID dari fitur clustering.

In [4]:
X, z = make_blobs(n_samples=800, centers=4, n_features=4,
                 cluster_std=[0.8, 1.0, 1.2, 0.9], random_state=SEED)
X = (X-X.min(axis=0)) / (X.max(axis=0)-X.min(axis=0))
perkerasan = pd.DataFrame({
    "id_segmen": [f"S{i:04d}" for i in range(len(z))],
    "retak_persen": (X[:, 0]*12).round(2),
    "kedalaman_alur_mm": (X[:, 1]*25).round(2),
    "IRI_m_km": (X[:, 2]*8).round(2),
    "lalu_lintas_tahunan": (X[:, 3]*2.0e6).round(0),
    "label_pembangkit": z})
perkerasan.to_csv(out_dir / "data_sintesis_perkerasan.csv", index=False)
perkerasan.head()

,id_segmen,retak_persen,kedalaman_alur_mm,IRI_m_km,lalu_lintas_tahunan,label_pembangkit
0,S0000,2.90,22.62,3.60,277903.0,2
1,S0001,9.40,7.99,5.36,1052827.0,1
2,S0002,7.86,5.86,2.96,1716541.0,3
3,S0003,8.75,1.64,3.55,1079911.0,0
4,S0004,4.22,23.24,5.05,58746.0,2


## 4. Sinyal sintesis untuk latihan anomali
Flag simulasi menandai titik yang sengaja digeser. Skor anomali bukan diagnosis kerusakan.

In [5]:
q = 700
waktu = pd.date_range("2024-01-01", periods=q, freq="h")
suhu = 20 + 5*np.sin(np.arange(q)*2*np.pi/24) + rng.normal(0, .5, q)
getaran = .25 + .015*suhu + rng.normal(0, .04, q)
flag = rng.random(q) < .025
getaran[flag] += rng.uniform(.5, 1.4, flag.sum())
sensor = pd.DataFrame({"waktu": waktu, "suhu_C": suhu.round(2),
    "getaran_mm_s": getaran.round(3), "flag_pembangkit": flag})
sensor.to_csv(out_dir / "data_sintesis_sensor.csv", index=False)
sensor.head()

,waktu,suhu_C,getaran_mm_s,flag_pembangkit
0,2024-01-01 00:00:00,20.15,0.506,False
1,2024-01-01 01:00:00,21.60,0.623,False
2,2024-01-01 02:00:00,21.87,0.609,False
3,2024-01-01 03:00:00,23.38,0.604,False
4,2024-01-01 04:00:00,23.82,0.639,False


## Audit dan batas penggunaan
Keempat CSV tersimpan dalam `data_bab4` (dari akar repositori: `praktikum/bab04/data_bab4`). Pada klasifikasi, ID aset adalah metadata, bukan prediktor. Pada clustering, buang ID dan label pembangkit dari fitur. Tidak satu pun dataset mewakili beton, jembatan, perkerasan, atau sensor nyata.